In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Data = pd.read_csv("data.csv")



In [7]:
num_col = Data.select_dtypes(include= ["int64","float64"]).columns

In [12]:
"""
 In Excel when we plotted Box plot, min and max values are basically min and max value of dataset itself.
 In seaborn min and max are calculated on the basis of IQR (Inter-Quartile Range)
 IQR = Q3-Q1
 min_whisker = q1-(1.5*IQR)
 max_whisker = q3+(1.5*IQR)

"""

q1 = Data["bathrooms"].quantile(0.25)
q2 = Data["bathrooms"].quantile(0.5)
q3 = Data["bathrooms"].quantile(0.75)
IQR = q3-q1
ll = q1-(1.5*IQR)
ul = q3+(1.5*IQR)
ll,q1,q2,q3,ul

(np.float64(-2.0),
 np.float64(1.0),
 np.float64(2.0),
 np.float64(3.0),
 np.float64(6.0))

In [16]:
select_rows = Data[(Data["bathrooms"]<ll) | (Data["bathrooms"]>ul)]
(select_rows.shape[0]/Data.shape[0])*100

0.24704199713951372

In [5]:
def get_ll_ul(Data,col):
  q1 = Data[col].quantile(0.25)
  q3 = Data[col].quantile(0.75)
  IQR = q3-q1
  ll = q1-(1.5*IQR)
  ul = q3+(1.5*IQR)
  return ll,ul

In [18]:
for col in num_col:
  ll,ul = get_ll_ul(Data,col)
  select_rows = Data[(Data[col]<ll) | (Data[col]>ul)].shape[0]
  print(col, "-->",select_rows," ",(select_rows/Data.shape[0])*100)

area --> 490   6.371083084124302
beds --> 17   0.22103757638798596
bathrooms --> 19   0.24704199713951372
balconies --> 13   0.16902873488493045
area_rate --> 666   8.659472110258744
rent --> 713   9.270575997919646


In [19]:
Outlier_col = ['beds','balconies','bathrooms']
for col in Outlier_col:
  ll,ul = get_ll_ul(Data,col)
  Data = Data[(Data[col]>=ll) & (Data[col]<=ul)]
  Data.reset_index(drop = True, inplace = True)
  print(f"Outlier Removed from {col}")

Outlier Removed from beds
Outlier Removed from balconies
Outlier Removed from bathrooms


In [20]:
Data.shape

(7654, 10)

In [31]:
# Feature Engineering
Data[['house_type','beds']].sample(10)

,house_type,beds
4238,"2 BHK Flat for Rent in Empire Estate, Anand Na...",2
2153,"3 BHK Flat for Rent in Rustomjee Seasons, Kala...",3
3305,"2 BHK House for Rent in DDA Flats Munirka, Mun...",2
5069,"1 BHK Flat for Rent in Durga Vihar Devli, New ...",1
1304,1 BHK House for Rent in Sector 2 HSR Layout Ba...,1
7320,"4 BHK Flat for Rent in Block 5th Jayanagar, Ba...",4
4914,"1 BHK Flat for Rent in Aga Nagar, Vadgaonsheri...",1
877,"1 BHK Flat for Rent in Mayfair The View, Vikhr...",1
3456,"3 BHK Flat for Rent in Amanora Trendy Homes, H...",3
6201,2 BHK Flat for Rent in Indra Enclave Sainik Fa...,2


In [29]:
Data['property_type'] = Data['house_type'].apply(lambda x: (x.split()[2]))
Data['property_type'].value_counts()  # value_counts will give u both catgeory types and it also counts how many times that particular catgeory has been occurred in the dataset.

,count
property_type,
Flat,5878
House,1547
Villa,229


In [32]:
Data.columns

Index(['house_type', 'locality', 'city', 'area', 'beds', 'bathrooms',
       'balconies', 'furnishing', 'area_rate', 'rent', 'bedrooms',
       'property_type'],
      dtype='object')

In [34]:
Data.drop(columns = ['house_type'],axis=1,inplace = True)

In [37]:
Data.sample(5)

,locality,city,area,beds,bathrooms,balconies,furnishing,area_rate,rent,bedrooms,property_type
3351,137,4,720.0,2,2,3,2,45.0,32500.0,2,0
2075,1344,3,360.0,1,1,0,0,42.0,15000.0,1,0
2195,1889,3,900.0,2,2,0,2,13.0,11500.0,2,1
6479,1141,2,1440.0,3,3,0,2,9.0,13000.0,3,0
84,1092,0,800.0,2,2,2,2,38.0,30000.0,2,0


In [36]:
# Converting categorical col to numerical(Encoding)

from sklearn.preprocessing import LabelEncoder
categorical_col = ["locality","city","furnishing","property_type"]

for col in categorical_col:
  le = LabelEncoder()
  Data[col] = le.fit_transform(Data[col])